# 02 - Baseline: DINOv2 features -> logistic regression (local)

**Idea:** each clip is 30 vectors (one per frame). We squeeze them into ONE vector per clip in several ways ("feature groups"),
train a simple classifier on each group, and compare the cross-validated AUC. This tells us which kind of information carries the crash signal.

**Validation:** 5 folds from `make_folds()`. Twins (same `group_id`) are always in the same fold, so no leakage.
**Runs locally** on saved features (CPU is fine, it takes about a minute).

In [1]:
import sys
sys.path.append("..")                       # so we can import src/common.py
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from src.common import ART, SEED, load_labels, make_folds, rank_average

# Load features made on Kaggle: shape (clips, 30 frames, 1536)
train_emb = np.load(ART / "features/dinov2b_train.npy").astype(np.float32)
test_emb = np.load(ART / "features/dinov2b_test.npy").astype(np.float32)

lab = load_labels().merge(make_folds(), on="clip_id")     # clip_id, label, group_id, fold (csv order kept)
y, fold = lab["label"].values, lab["fold"].values
print(train_emb.shape, test_emb.shape, "| crash rate:", y.mean())

(3000, 30, 1536) (1750, 30, 1536) | crash rate: 0.5


## 1. Build feature groups
Each group turns (30, 1536) per clip into one vector. Think of them as different "questions" we ask about a clip:
- **mean / std / max over time**: what does the clip look like on average? (scene/style information)
- **diff mean / diff max**: how much do the frame vectors change from one frame to the next? (change = crash signal, but also risky under sharpness shift)
- **curve (raw)**: ONE number per time step = how far the embedding moved between frame t and t+2 (28 numbers). Stride 2 cancels the test zigzag.
- **curve (shape)**: the same curve divided by its own average -> only the *shape* of the change (does motion build up?), not its size. Sharpness cancels out.

In [2]:
def make_groups(e):
    d1 = np.abs(e[:, 1:] - e[:, :-1])                              # change between neighbouring frames, per feature
    step2 = np.linalg.norm(e[:, 2:] - e[:, :-2], axis=-1)           # (clips, 28): size of the jump from frame t to t+2
    shape = step2 / (step2.mean(1, keepdims=True) + 1e-6)           # same curve with its average size removed
    return {
        "mean over time": e.mean(1),
        "std over time": e.std(1),
        "max over time": e.max(1),
        "diff mean": d1.mean(1),
        "diff max": d1.max(1),
        "curve raw (28)": step2,
        "curve shape (28)": shape,
    }

groups_train, groups_test = make_groups(train_emb), make_groups(test_emb)
groups_train["ALL pooled"] = np.concatenate([groups_train[k] for k in ["mean over time", "std over time", "max over time", "diff mean", "diff max"]], axis=1)
groups_test["ALL pooled"] = np.concatenate([groups_test[k] for k in ["mean over time", "std over time", "max over time", "diff mean", "diff max"]], axis=1)
for k, v in groups_train.items():
    print(f"{k:18s} {v.shape}")

mean over time     (3000, 1536)
std over time      (3000, 1536)
max over time      (3000, 1536)
diff mean          (3000, 1536)
diff max           (3000, 1536)
curve raw (28)     (3000, 28)
curve shape (28)   (3000, 28)
ALL pooled         (3000, 7680)


## 2. Grouped cross-validation
For every fold: train on 4 folds, predict the held-out fold. Stitching the held-out predictions together gives out-of-fold (OOF) predictions for all 3000 clips,
and one AUC on those. `StandardScaler` puts all features on the same scale; a small `C` = strong regularisation (important with 3000 clips and many features).

In [3]:
def cv_auc(X, Xtest=None, C=0.01):
    oof = np.zeros(len(y))
    test_pred = np.zeros(len(Xtest)) if Xtest is not None else None
    for f in range(5):
        tr, va = fold != f, fold == f
        model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000))
        model.fit(X[tr], y[tr])
        oof[va] = model.predict_proba(X[va])[:, 1]
        if Xtest is not None:
            test_pred += model.predict_proba(Xtest)[:, 1] / 5      # average the 5 fold models on test
    return roc_auc_score(y, oof), oof, test_pred

results, oofs, tests = {}, {}, {}
for name in groups_train:
    auc, oofs[name], tests[name] = cv_auc(groups_train[name], groups_test[name])
    results[name] = auc
    print(f"{name:18s} CV AUC = {auc:.4f}")

mean over time     CV AUC = 0.8871


std over time      CV AUC = 0.8991


max over time      CV AUC = 0.9056


diff mean          CV AUC = 0.7714


diff max           CV AUC = 0.7383
curve raw (28)     CV AUC = 0.9296
curve shape (28)   CV AUC = 0.9368


ALL pooled         CV AUC = 0.9432


## 3. Sanity check (non-negotiable): shuffle the labels
If we shuffle the labels, no real signal exists, so AUC must be about 0.50. If it is much higher, something leaks. We shuffle labels *within each group* so twins stay balanced.

In [4]:
rng = np.random.default_rng(SEED)
y_real = y.copy()
y = lab.groupby("group_id")["label"].transform(lambda s: rng.permutation(s.values)).values   # shuffled labels
auc_shuffled, _, _ = cv_auc(groups_train["ALL pooled"])
y = y_real                                                                                     # restore real labels
print(f"shuffled-label CV AUC = {auc_shuffled:.4f}  (should be about 0.50)")

shuffled-label CV AUC = 0.5208  (should be about 0.50)


## 4. Save the best OOF/test predictions and show the summary

In [5]:
summary = pd.Series(results).sort_values(ascending=False)
print(summary.round(4).to_string())

best = summary.index[0]
np.save(ART / "oof_dinov2_lr.npy", oofs["ALL pooled"])             # the standard "all pooled" baseline
np.save(ART / "test_dinov2_lr.npy", tests["ALL pooled"])
print("saved baseline preds for: ALL pooled | best single group:", best)

ALL pooled          0.9432
curve shape (28)    0.9368
curve raw (28)      0.9296
max over time       0.9056
std over time       0.8991
mean over time      0.8871
diff mean           0.7714
diff max            0.7383
saved baseline preds for: ALL pooled | best single group: ALL pooled
